# 1.4 规约算子

前两节分别学习了逐元素计算和矩阵乘法。本节学习第三类基础计算：规约。逐元素计算通常是“一个位置对应一个位置”，矩阵乘法是“行和列做乘加”，而规约是“沿某个维度把一组元素合成更少的元素”。

这个区别要和 shape 一起理解。逐元素计算大多保持 shape，矩阵乘法会把 `[M, K] @ [K, N]` 变成 `[M, N]`，规约则会沿 `dim` 指定的方向缩小 shape。例如 `[2, 3]` 沿最后一维求和会得到 `[2]`，如果设置 `keepdim=True` 则得到 `[2, 1]`。

本节围绕四个问题展开：`dim=-1` 到底指哪个方向，`keepdim=True` 为什么常用于后续广播，`sum/amax/amin` 这类规约和 `maximum/minimum` 这类逐元素比较有什么区别，以及规约如何和逐元素计算组合成归一化、中心化和 Softmax 这类常见模式。


---
## 前置说明

本节在第一个代码单元格中集中放置环境准备代码（import、设备选择、`RUN_MODE` 等）。后续每个可运行模块在定义 kernel 前只调用 `reset_pypto_notebook_state()` 重置编译状态，不再重复完整的环境初始化。

环境准备的具体含义参见 `01.01_chapter_intro.ipynb` §1。


## 直观理解：规约就是“把一组数字合成一个数字”

规约听起来像一个专业词，但可以先理解为“汇总”。例如一行有 3 个数字：

```python
[1, 2, 3]
```

如果做求和规约，就会得到：

```python
1 + 2 + 3 = 6
```

也就是说，规约会把多个数字合成一个数字。`sum` 是求和，`amax` 是取最大值，`amin` 是取最小值。


## 2. 什么是规约

规约是把某个维度上的多个元素合并成一个元素。常见规约包括：

| 规约类型 | 含义 |
| --- | --- |
| `sum` | 求和 |
| `amax` | 求最大值 |
| `amin` | 求最小值 |
| `mean` | 求平均值 |

以二维 Tensor 为例：

```python
x = [
    [1, 2, 3],
    [4, 5, 6],
]
```

如果沿最后一个维度求和，每一行会被规约成一个值，结果是 `[6, 15]`。


## 2.1 常见规约 API 与逐元素比较 API

PyPTO 中常见的规约 API 包括 `sum`、`amax` 和 `amin`。它们都需要关注 `dim` 和 `keepdim`，因为它们会沿某个维度把多个元素合并起来。

| API | 类型 | shape 变化 |
| --- | --- | --- |
| `pypto.sum(a, dim, keepdim)` | reduction | 沿 `dim` 求和 |
| `pypto.amax(a, dim, keepdim)` | reduction | 沿 `dim` 取最大值 |
| `pypto.amin(a, dim, keepdim)` | reduction | 沿 `dim` 取最小值 |
| `pypto.maximum(a, b)` | elementwise | 对应元素取较大值，输出 shape 与广播后的输入一致 |
| `pypto.minimum(a, b)` | elementwise | 对应元素取较小值，输出 shape 与广播后的输入一致 |

这里最容易混淆的是 `amax` 和 `maximum`。`amax(a, dim=-1)` 是“在一行内部找最大值”，会减少或压缩某个维度；`maximum(a, b)` 是“两个 Tensor 对应位置比较”，仍然属于逐元素计算。


## 2.2 elementwise 与 reduction 的边界

`elementwise` 和 `reduction` 是两类最基础的 Tensor 计算。可以用“一个位置看几个输入值”来区分它们。

`elementwise` 通常是每个输出位置只看输入中对应位置的元素。例如：

```python
out[:] = pypto.add(a, b)
out[:] = pypto.maximum(a, b)
```

`reduction` 则是一个输出位置要看输入中一组元素。例如：

```python
out[:] = pypto.sum(a, dim=-1, keepdim=False)
out[:] = pypto.amax(a, dim=-1, keepdim=False)
```

所以判断一个 API 属于哪类，不只看名字像不像“最大值”，还要看它是否沿某个 `dim` 汇总。`maximum(a, b)` 没有 `dim`，它比较的是对应位置；`amax(a, dim=-1)` 有 `dim`，它汇总的是一整行或某个维度上的一组值。


## 3. dim=-1 的含义

`dim=-1` 是初学者非常容易误解的写法。它不是“最后一行”，而是“最后一个维度”。

对于形状 `[M, N]` 的二维 Tensor：

- `dim=0` 表示第 0 个维度，也就是行数方向。
- `dim=1` 表示第 1 个维度，也就是列方向。
- `dim=-1` 表示最后一个维度，对二维 Tensor 来说等价于 `dim=1`。

所以 `pypto.sum(x, dim=-1)` 表示把每一行内部的元素加起来。


### 用“方向”理解 dim

对于二维 Tensor，可以把它想象成二维数据：

```python
[[1, 2, 3],
 [4, 5, 6]]
```

- 沿 `dim=0` 看，是“竖着看”，会跨行处理。
- 沿 `dim=1` 看，是“横着看”，会在每一行内部处理。
- `dim=-1` 表示最后一个维度。二维 Tensor的最后一个维度就是列方向，所以它也是“横着看”。

因此 `sum(x, dim=-1)` 会得到每一行的和，而不是最后一行的和。


## 4. keepdim=True 的作用

如果 `x` 的形状是 `[2, 3]`：

```python
pypto.sum(x, dim=-1, keepdim=False)
```

输出形状是 `[2]`。它表示每一行得到一个数字，但这个结果已经不再是二维列向量。

而：

```python
pypto.sum(x, dim=-1, keepdim=True)
```

输出形状是 `[2, 1]`。它仍然保留“每一行对应一个汇总值”的二维结构。

保留这个维度的主要价值是方便广播回原 shape。例如：

```python
out = x / row_sum
```

如果 `row_sum` 形状是 `[2, 1]`，它可以广播到 `[2, 3]`，表示每一行都除以自己这一行的和。后面的归一化、中心化和 Softmax 都依赖这个思路。


### 为什么保留维度对初学者很重要

假设原始 Tensor 是 2 行 3 列，也就是 shape `[2, 3]`。

如果每一行求和但不保留维度，结果像这样：

```python
[6, 15]        # shape 是 [2]
```

如果保留维度，结果像这样：

```python
[[6],
 [15]]         # shape 是 [2, 1]
```

第二种看起来多了一层括号，但它更适合后续和原来的 `[2, 3]` 二维数据做广播计算。可以理解为“每一行的汇总结果仍然站在自己那一行旁边”。这也是为什么很多归一化类算子都会写 `keepdim=True`。


## 5. 编写按行求和算子

下面先实现最简单的按行求和。这里 `keepdim=False`，所以输出是一维 Tensor。


## 5.1 按行求和算子规格

| 项目 | 说明 |
| --- | --- |
| 数学表达式 | `out = sum(x, dim=-1)` |
| 输入 `x` | FP32 Tensor，示例 shape 为 `[2, 3]` |
| 输出 `out` | FP32 Tensor，shape 为 `[2]` |
| 规约维度 | `dim=-1`，即最后一个维度 |
| `keepdim` | `False`，不保留被规约维度 |

这个例子用于说明规约会减少 Tensor 的维度。

在 Notebook 中，每个可运行模块都先清理 PyPTO 的记录状态，再定义当前模块的 JIT kernel。这样一个模块的失败不会影响下一个模块的验证。


In [ ]:
import os

# 环境准备：参见 01.01_chapter_intro.ipynb §1 的详细说明
# ASCEND_RT_VISIBLE_DEVICES 必须在 import torch 之前设置才生效。
# 如需指定其他 NPU，请在启动 Notebook 前设置 ASCEND_RT_VISIBLE_DEVICES。
# os.environ.setdefault("ASCEND_RT_VISIBLE_DEVICES", "13")

import torch
import pypto
from numpy.testing import assert_allclose

try:
    import torch_npu  # noqa: F401
except Exception:
    torch_npu = None


def reset_pypto_notebook_state():
    "Clean stale PyPTO recording state before defining a JIT kernel."
    try:
        pypto.reset()
    except Exception:
        pass

    try:
        from pypto._controller import Controller
        Controller.end_function()
    except Exception:
        pass


def get_device():
    if torch_npu is None:
        return "cpu"

    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    try:
        torch.npu.set_device(device_id)
    except Exception as exc:
        print("NPU device is not ready:", exc)
        return "cpu"
    return f"npu:{device_id}"


reset_pypto_notebook_state()
device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

# print("ASCEND_RT_VISIBLE_DEVICES:", os.environ.get("ASCEND_RT_VISIBLE_DEVICES", "<未设置>"))
print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<未设置，默认 0>"))
print("device:", device)

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def row_sum_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out.move(pypto.sum(x, dim=-1, keepdim=False))


def main_row_sum():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32, device=device)
    out = torch.empty((2,), dtype=torch.float32, device=device)

    row_sum_kernel(x, out)
    ref = torch.sum(x, dim=-1)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("row_sum_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("输入:", x.cpu())
    print("输出:", out.cpu())
    print("参考:", ref.cpu())
    print("最大误差:", max_diff)


main_row_sum()


ASCEND_RT_VISIBLE_DEVICES: 13
TILE_FWK_DEVICE_ID: <未设置，默认 0>
device: npu:0
row_sum_kernel 验证通过
device: npu:0 run_mode: 0
输入: tensor([[1., 2., 3.],
        [4., 5., 6.]])
输出: tensor([ 6., 15.])
参考: tensor([ 6., 15.])
最大误差: 0.0


`row_sum_kernel` 沿最后一维求和。二维输入可以理解成一张表，`dim=-1` 表示把每一行内部的元素加起来，因此输出只保留每一行的求和结果。


### 代码细节解释

- `pypto.sum(x, dim=-1, keepdim=False)` 沿最后一个维度求和。
- 输入 shape `[2, 3]` 会变成输出 shape `[2]`。
- 这里使用 `set_vec_tile_shapes(2, 8)`，不是因为输入有 8 列，而是因为规约最后一维的 Tile 需要满足 32Byte 对齐。
- 当前 dtype 是 FP32，每个元素 4Byte，因此最后一维 Tile 至少按 8 个元素对齐，`8 * 4Byte = 32Byte`。
- 输入实际 shape 仍然是 `[2, 3]`，多出来的 Tile 宽度是执行组织上的对齐要求，不改变数学输入和输出。

这个例子刻意使用具体数字输入，目的不是追求大规模计算，而是让 shape 变化一眼可见：两行数据规约后仍然对应两个输出值。同时也可以看到，Tile Shape 不一定等于 Tensor Shape。规约用例中常用 `[8, 8, ...]` 这样的 Tile 设置，是为了让各维度尤其是最后一维满足底层执行对齐要求。


### 把 row_sum_kernel 解释为自然语言

这段代码做的事是：

“给我一张二维数字表 `x`，我把每一行里的数字加起来，然后把每一行的求和结果放进 `out`。”

如果输入是：

```python
[[1, 2, 3],
 [4, 5, 6]]
```

输出就是：

```python
[6, 15]
```


上面的完整单元用来验证 `row_sum_kernel`。它先在当前设备上构造一个二维输入，再提前分配一维输出 Tensor 并调用 PyPTO kernel。随后用普通 PyTorch 写出等价的按行求和参考结果，打印输入、输出、参考值和最大误差；最后的 `assert_close` 是正式检查。


**预期输出说明**

运行成功后，会看到输入 `[[1, 2, 3], [4, 5, 6]]`，输出应接近 `[6, 15]`，并显示最大误差。


## 5.2 最大值和最小值规约：amax 与 amin

求和不是唯一的规约。`pypto.amax` 和 `pypto.amin` 也会沿某个维度把一组元素合成一个元素，只是合成方式从“相加”变成“取最大值”或“取最小值”。

对于输入：

```python
[[1, 2, 3],
 [4, 5, 6]]
```

沿 `dim=-1` 做 `amax` 得到 `[3, 6]`，沿 `dim=-1` 做 `amin` 得到 `[1, 4]`。它们和 `sum` 一样都会改变 shape：`[2, 3] -> [2]`。

In [27]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def row_extrema_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    max_out: pypto.Tensor([], pypto.DT_FP32),
    min_out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    max_out.move(pypto.amax(x, dim=-1, keepdim=False))
    min_out.move(pypto.amin(x, dim=-1, keepdim=False))


def main_row_extrema():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32, device=device)
    max_out = torch.empty((2,), dtype=torch.float32, device=device)
    min_out = torch.empty((2,), dtype=torch.float32, device=device)

    row_extrema_kernel(x, max_out, min_out)
    max_ref = torch.amax(x, dim=-1)
    min_ref = torch.amin(x, dim=-1)

    max_diff = max((max_out - max_ref).abs().max().item(), (min_out - min_ref).abs().max().item())
    torch.testing.assert_close(max_out, max_ref, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(min_out, min_ref, rtol=1e-3, atol=1e-3)
    print("row_extrema_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("输入:", x.cpu())
    print("每行最大值:", max_out.cpu(), "参考:", max_ref.cpu())
    print("每行最小值:", min_out.cpu(), "参考:", min_ref.cpu())
    print("最大误差:", max_diff)


main_row_extrema()


row_extrema_kernel 验证通过
device: npu:0 run_mode: 0
输入: tensor([[1., 2., 3.],
        [4., 5., 6.]])
每行最大值: tensor([3., 6.]) 参考: tensor([3., 6.])
每行最小值: tensor([1., 4.]) 参考: tensor([1., 4.])
最大误差: 0.0


`row_extrema_kernel` 同时计算每一行的最大值和最小值。这里依然使用 `set_vec_tile_shapes(2, 8)`，原因和 `sum` 一样：FP32 规约最后一维 Tile 需要满足 32Byte 对齐。

这段代码也展示了 PyPTO kernel 可以有多个输出 Tensor。`max_out` 保存 `amax` 结果，`min_out` 保存 `amin` 结果；host 侧分别用 `torch.amax` 和 `torch.amin` 作为参考结果。

**预期输出说明**

运行成功后，会看到 `row_extrema_kernel 验证通过`，并打印每行最大值 `[3, 6]`、每行最小值 `[1, 4]` 以及最大误差。

## 5.3 逐元素比较：maximum 与 minimum

`maximum/minimum` 和 `amax/amin` 名字相近，但计算类型不同。

`amax/amin` 是规约：一个输出元素来自输入中一组元素，因此需要 `dim`。

`maximum/minimum` 是逐元素比较：一个输出元素来自两个输入 Tensor 的对应位置，因此不需要 `dim`，输出 shape 和广播后的输入 shape 一致。

例如：

```python
a = [[1, 2, 3], [4, 5, 6]]
b = [[0, 9, 2], [1, 3, 10]]
maximum(a, b) = [[1, 9, 3], [4, 5, 10]]
minimum(a, b) = [[0, 2, 2], [1, 3, 6]]
```

In [28]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def elementwise_compare_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    max_out: pypto.Tensor([], pypto.DT_FP32),
    min_out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    max_out.move(pypto.maximum(a, b))
    min_out.move(pypto.minimum(a, b))


def main_elementwise_compare():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32, device=device)
    b = torch.tensor([[0, 9, 2], [1, 3, 10]], dtype=torch.float32, device=device)
    max_out = torch.empty_like(a)
    min_out = torch.empty_like(a)

    elementwise_compare_kernel(a, b, max_out, min_out)
    max_ref = torch.maximum(a, b)
    min_ref = torch.minimum(a, b)

    max_diff = max((max_out - max_ref).abs().max().item(), (min_out - min_ref).abs().max().item())
    torch.testing.assert_close(max_out, max_ref, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(min_out, min_ref, rtol=1e-3, atol=1e-3)
    print("elementwise_compare_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("a:", a.cpu())
    print("b:", b.cpu())
    print("maximum 输出:", max_out.cpu())
    print("minimum 输出:", min_out.cpu())
    print("最大误差:", max_diff)


main_elementwise_compare()


elementwise_compare_kernel 验证通过
device: npu:0 run_mode: 0
a: tensor([[1., 2., 3.],
        [4., 5., 6.]])
b: tensor([[ 0.,  9.,  2.],
        [ 1.,  3., 10.]])
maximum 输出: tensor([[ 1.,  9.,  3.],
        [ 4.,  5., 10.]])
minimum 输出: tensor([[0., 2., 2.],
        [1., 3., 6.]])
最大误差: 0.0


`elementwise_compare_kernel` 没有使用 `dim`，因为它不是规约。它只是在每个位置比较 `a` 和 `b`，分别写出较大值和较小值。

这个模块的重点是建立边界感：看到 `amax/amin + dim`，要想到 reduction 和 shape 变化；看到 `maximum/minimum(a, b)`，要想到 elementwise 和对应位置比较。

**预期输出说明**

运行成功后，会看到 `elementwise_compare_kernel 验证通过`，并打印 `maximum` 与 `minimum` 的二维输出。输出 shape 与输入 shape 相同。

## 6. 编写按行归一化算子

接下来实现一个更有实际意义的例子：按行归一化。

计算流程如下：

```python
row_sum = sum(x, dim=-1, keepdim=True)
out = x / row_sum
```

这段代码由一个规约和一个逐元素除法组成。`row_sum` 是规约结果，`x / row_sum` 是逐元素计算，并依赖广播。

这里必须注意 `keepdim=True`。如果不保留维度，`row_sum` 是 `[M]`；保留维度后，`row_sum` 是 `[M, 1]`，后续和 `[M, N]` 的 `x` 做广播时更直观。


## 6.1 按行归一化算子规格

| 项目 | 说明 |
| --- | --- |
| 数学表达式 | `out = x / sum(x, dim=-1, keepdim=True)` |
| 输入 `x` | FP32 Tensor，示例 shape 为 `[8, 8]` |
| 中间变量 `row_sum` | shape 为 `[8, 1]` |
| 输出 `out` | shape 为 `[8, 8]` |
| `keepdim=True` | 保留规约维度，方便广播 |

这里的关键是 `row_sum` 的 shape。它保留为 `[8, 1]` 后，可以自然广播到 `[8, 8]`。


In [29]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def row_normalize_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    row_sum = pypto.sum(x, dim=-1, keepdim=True)
    out.move(x / row_sum)


def main_row_normalize():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.rand((8, 8), dtype=torch.float32, device=device) + 0.01
    out = torch.empty_like(x)

    row_normalize_kernel(x, out)
    ref = x / torch.sum(x, dim=-1, keepdim=True)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("row_normalize_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("输入 shape:", tuple(x.shape), "输出 shape:", tuple(out.shape))
    print("最大误差:", max_diff)
    print("输出前 2x4:", out[:2, :4].cpu())
    print("每行求和:", out.sum(dim=-1).cpu())


main_row_normalize()


row_normalize_kernel 验证通过
device: npu:0 run_mode: 0
输入 shape: (8, 8) 输出 shape: (8, 8)
最大误差: 0.0
输出前 2x4: tensor([[0.0639, 0.1150, 0.1712, 0.1281],
        [0.1305, 0.0867, 0.1671, 0.1175]])
每行求和: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


`row_normalize_kernel` 先计算每一行的和，并用 `keepdim=True` 保留成 `[batch, 1]`，再让原始输入除以这一列和。这样得到的输出仍是原 shape，但每一行加起来接近 1。

从计算类型看，`pypto.sum` 是 reduction，`x / row_sum` 是 elementwise。一个 kernel 里可以把它们串起来，但阅读时要分清哪一步改变 shape，哪一步保持 shape。


上面的完整单元用来验证 `row_normalize_kernel`。它先构造一个正数输入矩阵，再调用 PyPTO kernel 得到按行归一化结果。PyTorch 参考实现写成 `x / torch.sum(x, dim=-1, keepdim=True)`，最后打印 shape、最大误差、输出样例和每行求和结果。


**预期输出说明**

运行成功后，会打印输出前几项和每行求和结果。每行求和应接近 1。


## 7. 从 Softmax 看规约的价值

Softmax 是规约和逐元素计算组合的典型例子：

```python
row_max = pypto.amax(x, dim=-1, keepdim=True)
shifted = x - row_max
exp = pypto.exp(shifted)
esum = pypto.sum(exp, dim=-1, keepdim=True)
out = exp / esum
```

这里有两次规约：

1. `amax` 得到每一行最大值，用于数值稳定。
2. `sum` 得到每一行指数值之和，用于归一化。

也有多次逐元素计算：

1. `x - row_max`
2. `exp(...)`
3. `exp / esum`

`row_max` 和 `esum` 都设置 `keepdim=True`，是为了让它们能自然广播回原来的 `[batch, hidden]` 形状。复杂算子不是突然出现的新东西，而是 shape、broadcast、elementwise、reduction 和验证逻辑组合在一起。


## 8. 规约 API 全量覆盖

前面几节已经讲清楚规约的基本概念。下面补充 `sum`、`amax`、`amin` 在不同 `dim` 和 `keepdim` 设置下的完整示例，以及 `maximum/minimum` 逐元素比较的验证。阅读时重点看三件事：规约沿哪个维度发生、`keepdim` 是否保留被规约维度、输出 shape 如何变化。

In [30]:
def check_close(name, actual, expected, rtol=1e-3, atol=1e-3):
    assert_allclose(actual.cpu().numpy(), expected.cpu().numpy(), rtol=rtol, atol=atol)
    print(f"{name} verified")
    print("  output shape:", tuple(actual.shape))
    print("  expected shape:", tuple(expected.shape))
    print("  output:", actual.cpu())

### 8.1 sum、amax、amin 的通用 kernel

`reduce_ops.py` 里把 `sum_op`、`amax_op` 和 `amin_op` 写成 Python wrapper，并在 wrapper 内部根据 `dim/keepdim` 定义 JIT kernel。Notebook 中为了便于阅读，直接列出常用的固定维度 kernel。

In [31]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sum_last_keep_false_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.sum(a, dim=-1, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sum_last_keep_true_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.sum(a, dim=-1, keepdim=True)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sum_dim0_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8, 8)
    out[:] = pypto.sum(a, dim=0, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sum_dim1_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8, 8)
    out[:] = pypto.sum(a, dim=1, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def sum_dim2_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8, 8)
    out[:] = pypto.sum(a, dim=2, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amax_last_keep_false_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.amax(a, dim=-1, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amax_last_keep_true_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.amax(a, dim=-1, keepdim=True)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amax_dim0_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8, 8)
    out[:] = pypto.amax(a, dim=0, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amax_dim1_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8, 8)
    out[:] = pypto.amax(a, dim=1, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amax_dim2_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8, 8)
    out[:] = pypto.amax(a, dim=2, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amin_last_keep_false_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.amin(a, dim=-1, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amin_last_keep_true_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.amin(a, dim=-1, keepdim=True)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amin_dim0_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8, 8)
    out[:] = pypto.amin(a, dim=0, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amin_dim1_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8, 8)
    out[:] = pypto.amin(a, dim=1, keepdim=False)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def amin_dim2_kernel(a: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8, 8)
    out[:] = pypto.amin(a, dim=2, keepdim=False)


### 8.2 sum：basic 与不同维度

下面覆盖 `sum::test_sum_basic` 和 `sum::test_sum_different_dimensions`。

In [32]:
def test_sum_examples():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32, device=device)

    out = torch.empty((2,), dtype=torch.float32, device=device)
    sum_last_keep_false_kernel(a, out)
    check_close("sum::test_sum_basic keepdim=False", out, torch.sum(a, dim=-1, keepdim=False))

    out = torch.empty((2, 1), dtype=torch.float32, device=device)
    sum_last_keep_true_kernel(a, out)
    check_close("sum::test_sum_basic keepdim=True", out, torch.sum(a, dim=-1, keepdim=True))

    x = torch.tensor([
        [[1, 2, 3], [4, 5, 6]],
        [[7, 8, 9], [10, 11, 12]],
    ], dtype=torch.float32, device=device)

    out = torch.empty((2, 3), dtype=torch.float32, device=device)
    sum_dim0_kernel(x, out)
    check_close("sum::test_sum_different_dimensions dim=0", out, torch.sum(x, dim=0))

    out = torch.empty((2, 3), dtype=torch.float32, device=device)
    sum_dim1_kernel(x, out)
    check_close("sum::test_sum_different_dimensions dim=1", out, torch.sum(x, dim=1))

    out = torch.empty((2, 2), dtype=torch.float32, device=device)
    sum_dim2_kernel(x, out)
    check_close("sum::test_sum_different_dimensions dim=2", out, torch.sum(x, dim=2))


test_sum_examples()


sum::test_sum_basic keepdim=False verified
  output shape: (2,)
  expected shape: (2,)
  output: tensor([ 6., 15.])
sum::test_sum_basic keepdim=True verified
  output shape: (2, 1)
  expected shape: (2, 1)
  output: tensor([[ 6.],
        [15.]])
sum::test_sum_different_dimensions dim=0 verified
  output shape: (2, 3)
  expected shape: (2, 3)
  output: tensor([[ 8., 10., 12.],
        [14., 16., 18.]])
sum::test_sum_different_dimensions dim=1 verified
  output shape: (2, 3)
  expected shape: (2, 3)
  output: tensor([[ 5.,  7.,  9.],
        [17., 19., 21.]])
sum::test_sum_different_dimensions dim=2 verified
  output shape: (2, 2)
  expected shape: (2, 2)
  output: tensor([[ 6., 15.],
        [24., 33.]])


### 8.3 amax 与 amin：basic 与不同维度

下面覆盖 `amax::test_amax_basic`、`amax::test_amax_different_dimensions`、`amin::test_amin_basic` 和 `amin::test_amin_different_dimensions`。

In [ ]:
def test_amax_amin_examples():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32, device=device)

    out = torch.empty((2,), dtype=torch.float32, device=device)
    amax_last_keep_false_kernel(a, out)
    check_close("amax::test_amax_basic keepdim=False", out, torch.amax(a, dim=-1, keepdim=False))

    out = torch.empty((2, 1), dtype=torch.float32, device=device)
    amax_last_keep_true_kernel(a, out)
    check_close("amax::test_amax_basic keepdim=True", out, torch.amax(a, dim=-1, keepdim=True))

    out = torch.empty((2,), dtype=torch.float32, device=device)
    amin_last_keep_false_kernel(a, out)
    check_close("amin::test_amin_basic keepdim=False", out, torch.amin(a, dim=-1, keepdim=False))

    out = torch.empty((2, 1), dtype=torch.float32, device=device)
    amin_last_keep_true_kernel(a, out)
    check_close("amin::test_amin_basic keepdim=True", out, torch.amin(a, dim=-1, keepdim=True))

    x = torch.tensor([
        [[1, 2, 3], [4, 5, 6]],
        [[7, 8, 9], [10, 11, 12]],
    ], dtype=torch.float32, device=device)

    out = torch.empty((2, 3), dtype=torch.float32, device=device)
    amax_dim0_kernel(x, out)
    check_close("amax::test_amax_different_dimensions dim=0", out, torch.amax(x, dim=0))

    out = torch.empty((2, 3), dtype=torch.float32, device=device)
    amax_dim1_kernel(x, out)
    check_close("amax::test_amax_different_dimensions dim=1", out, torch.amax(x, dim=1))

    out = torch.empty((2, 2), dtype=torch.float32, device=device)
    amax_dim2_kernel(x, out)
    check_close("amax::test_amax_different_dimensions dim=2", out, torch.amax(x, dim=2))

    out = torch.empty((2, 3), dtype=torch.float32, device=device)
    amin_dim0_kernel(x, out)
    check_close("amin::test_amin_different_dimensions dim=0", out, torch.amin(x, dim=0))

    out = torch.empty((2, 3), dtype=torch.float32, device=device)
    amin_dim1_kernel(x, out)
    check_close("amin::test_amin_different_dimensions dim=1", out, torch.amin(x, dim=1))

    out = torch.empty((2, 2), dtype=torch.float32, device=device)
    amin_dim2_kernel(x, out)
    check_close("amin::test_amin_different_dimensions dim=2", out, torch.amin(x, dim=2))


test_amax_amin_examples()


amax::test_amax_basic keepdim=False verified
  output shape: (2,)
  expected shape: (2,)
  output: tensor([3., 6.])
amax::test_amax_basic keepdim=True verified
  output shape: (2, 1)
  expected shape: (2, 1)
  output: tensor([[3.],
        [6.]])


### 8.4 maximum 与 minimum：逐元素比较

`maximum/minimum` 和 `amax/amin` 名字相近，但不是规约。它们比较两个输入 Tensor 的对应位置，输出 shape 与输入 shape 一致。下面覆盖 `maximum::test_maximum_basic` 和 `minimum::test_minimum_basic`。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def maximum_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.maximum(a, b)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def minimum_kernel(a: pypto.Tensor([], pypto.DT_FP32), b: pypto.Tensor([], pypto.DT_FP32), out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    out[:] = pypto.minimum(a, b)


def test_maximum_minimum_examples():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32, device=device)
    b = torch.tensor([[0, 9, 2], [1, 3, 10]], dtype=torch.float32, device=device)

    out = torch.empty_like(a)
    maximum_kernel(a, b, out)
    check_close("maximum::test_maximum_basic", out, torch.maximum(a, b))

    out = torch.empty_like(a)
    minimum_kernel(a, b, out)
    check_close("minimum::test_minimum_basic", out, torch.minimum(a, b))


test_maximum_minimum_examples()


### 8.5 规约 API 小结

本节完整覆盖了 `reduce_ops.py` 中的 beginner 示例。`sum/amax/amin` 是 reduction，需要看 `dim` 和 `keepdim`；`maximum/minimum` 是 elementwise，不沿维度汇总。

如果看到输出 shape 变小，通常说明发生了规约；如果输出 shape 与输入 shape 保持一致，通常是在做逐元素比较或逐元素计算。

## 9. 课后练习

练习目标是实现一个按最后一维计算均值并中心化的算子：

```python
mean = sum(x, dim=-1, keepdim=True) / x.shape[-1]
out = x - mean
```

中心化的意思是：每一行都减去这一行自己的均值。完成后，每一行输出的均值应该接近 0。

自测题：

1. `dim=-1` 表示最后一行吗？
2. 为什么归一化类算子经常设置 `keepdim=True`？
3. `x - mean` 是 elementwise 还是 reduction？
4. Softmax 中 `amax` 的作用是什么？

参考答案：

1. 不是，表示最后一个维度。
2. 为了保留被规约的维度，方便后续广播。
3. 是 elementwise。`mean` 来自 reduction，但减法本身是逐元素计算。
4. 用于提升数值稳定性，降低指数计算溢出风险。


## 10. 可运行课后练习：按行中心化

下面给出可直接验证的实现：

```python
mean = sum(x, dim=-1, keepdim=True) / 8
out = x - mean
```

这里示例输入 shape 固定为 `[8, 8]`，所以每行长度是 8。`mean` 的 shape 是 `[8, 1]`，可以广播回 `[8, 8]`。中心化之后，每一行的均值应接近 0。


In [ ]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def row_center_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(2, 8)
    row_sum = pypto.sum(x, dim=-1, keepdim=True)
    mean = row_sum / 8.0
    out.move(x - mean)


def main_row_center():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((8, 8), dtype=torch.float32, device=device)
    out = torch.empty_like(x)

    row_center_kernel(x, out)
    ref = x - torch.mean(x, dim=-1, keepdim=True)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("row_center_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("输出 shape:", tuple(out.shape), "最大误差:", max_diff)
    print("输出每行均值:", out.mean(dim=-1).cpu())
    print("输出前 2x4:", out[:2, :4].cpu())


main_row_center()


这个模块把“按行求和”继续推进到“按行求均值”。`row_sum` 的 shape 是 `[8, 1]`，除以每行长度 8 得到 `mean`，再让原始输入 `x` 减去 `mean`。由于 `mean` 保留了最后一维，`x - mean` 可以按行广播。

从概念关系上看，这个例子正好把本节的几个概念串起来：`pypto.sum` 是规约，`keepdim=True` 保留广播所需的维度，`row_sum / 8.0` 和 `x - mean` 都是逐元素计算。


验证逻辑使用 PyTorch 的 `torch.mean(x, dim=-1, keepdim=True)` 写出参考结果。中心化完成后，每一行输出的均值应该非常接近 0，因此代码会打印 `输出每行均值`。


**预期输出说明**

运行成功后，会看到 `row_center_kernel 验证通过`，并打印当前 device、run_mode、输出 shape、最大误差、每行均值和输出前几项。


核心语句是：

```python
row_sum = pypto.sum(x, dim=-1, keepdim=True)
mean = row_sum / 8.0
out.move(x - mean)
```


## 11. 本节小结

本节学习了规约算子的基本用法和全量 API 覆盖。理解 `dim` 和 `keepdim` 是后续学习 Softmax、RMSNorm、LayerNorm 的关键。规约要始终和 shape 变化一起看：`sum/amax/amin` 会沿某个维度汇总，`maximum/minimum` 等逐元素操作则依赖对应位置或广播。下一节会进一步学习 Tiling 和形状操作，理解 Tensor 在计算前后如何被组织。